## 🔧 Reward Fix - Position Holding Threshold

**Problem Found:**
- No rewards for holding positions with small profits (< 1.0%)
- Example: UnrPnL% = 0.11% → 0.0000 reward (should be positive!)

**Fix Applied:**
- Lowered threshold: 1.0% → **0.05%** (20x more sensitive)
- Tightened drawdown penalty: -3.0% → **-2.0%** (earlier warning)
- Now rewards even small profitable holds (+0.1 base per step)

**Expected After Fix:**
- Step with 0.11% profit → reward ≈ 0.13 (0.11×0.3 + 0.1)
- Step with 0.16% profit → reward ≈ 0.15
- Encourages holding winners even during small fluctuations

In [1]:
import numpy as np
import pandas as pd
from src.environments.simple_trading_env import SimpleTradingEnv
from src.utils.indicator_utils import add_indicators
import json

# Load data
symbol = 'BTCUSDT'
timeframe = '5m'
data_path = f'data/binance-{symbol}-{timeframe}.pkl'
df = pd.read_pickle(data_path)

print(f'Loaded {len(df)} rows for {symbol} {timeframe}')

df['date'] = pd.to_datetime(df['date_close'])
add_indicators(df)
df = df.dropna().reset_index(drop=True)

# Use recent data for testing
total_timesteps = 1000
test_data = df.iloc[-total_timesteps:]

# Create environment (no wrappers for direct testing)
env = SimpleTradingEnv(test_data, initial_balance=10000, device="cuda")

print(f"\n{'='*70}")
print("Environment initialized - Ready for action testing")
print(f"{'='*70}")
print(f"Initial Balance: ${env.initial_balance:,.2f}")
print(f"Action Space: {env.action_space}")
print(f"  Action format: [direction, risk_reward_ratio, atr_multiplier]")
print(f"  Direction: 0=HOLD, 1=LONG, 2=SHORT, 3=CLOSE")
print(f"  Risk-Reward: 0-9 (maps to 1.0x-10.0x)")
print(f"  ATR Multiplier: 0-9 (maps to 1.5-3.3)")
print(f"{'='*70}\n")

Loaded 264323 rows for BTCUSDT 5m

Environment initialized - Ready for action testing
Initial Balance: $10,000.00
Action Space: MultiDiscrete([4])
  Action format: [direction, risk_reward_ratio, atr_multiplier]
  Direction: 0=HOLD, 1=LONG, 2=SHORT, 3=CLOSE
  Risk-Reward: 0-9 (maps to 1.0x-10.0x)
  ATR Multiplier: 0-9 (maps to 1.5-3.3)


Environment initialized - Ready for action testing
Initial Balance: $10,000.00
Action Space: MultiDiscrete([4])
  Action format: [direction, risk_reward_ratio, atr_multiplier]
  Direction: 0=HOLD, 1=LONG, 2=SHORT, 3=CLOSE
  Risk-Reward: 0-9 (maps to 1.0x-10.0x)
  ATR Multiplier: 0-9 (maps to 1.5-3.3)



## Test 1: Random Actions

In [2]:
np.random.seed(42)
obs, _ = env.reset()

steps_to_run = 100
history_records = []

print(f"Running {steps_to_run} steps with RANDOM actions...")
print(f"{'='*70}\n")

for step in range(steps_to_run):
    action = env.action_space.sample()
    obs, reward, done, truncated, info = env.step(action)
    history_records.append(info)
    if done or truncated:
        print(f"Episode ended at step {step}")
        break

df_history = pd.DataFrame(history_records)
df_history['direction'] = df_history['action'].apply(lambda x: x[0] if isinstance(x, np.ndarray) else x)

# Summary
print("RANDOM ACTION TEST SUMMARY")
print(f"{'='*70}")
print(f"Total Steps: {len(df_history)}")
print(f"Total Reward: {df_history['reward'].sum():.4f} | Avg/Step: {df_history['reward'].mean():.4f}")
print(f"Final Equity: ${df_history['equity'].iloc[-1]:,.2f}")
print(f"P&L: ${df_history['equity'].iloc[-1] - df_history['equity'].iloc[0]:,.2f}")

# Action distribution
action_names = {0: 'HOLD', 1: 'LONG', 2: 'SHORT', 3: 'CLOSE'}
action_counts = df_history['direction'].value_counts().sort_index()
print(f"\nAction Distribution:")
for action_id, count in action_counts.items():
    print(f"  {action_names.get(action_id, action_id):>6}: {count:3d} ({count/len(df_history)*100:5.1f}%)")

# Reward by action
print(f"\nReward by Action:")
reward_summary = df_history.groupby('direction')['reward'].agg(['count', 'mean', 'sum']).round(4)
for idx, row in reward_summary.iterrows():
    print(f"  {action_names.get(idx, idx):>6}: count={row['count']:3.0f}, mean={row['mean']:7.4f}, sum={row['sum']:8.4f}")

print(f"{'='*70}\n")

Running 100 steps with RANDOM actions...

RANDOM ACTION TEST SUMMARY
Total Steps: 100
Total Reward: -1.3937 | Avg/Step: -0.0139
Final Equity: $9,397.93
P&L: $-595.21

Action Distribution:
    HOLD:  24 ( 24.0%)
    LONG:  15 ( 15.0%)
   SHORT:  26 ( 26.0%)
   CLOSE:  35 ( 35.0%)

Reward by Action:
    HOLD: count= 24, mean= 0.0000, sum=  0.0007
    LONG: count= 15, mean=-0.0190, sum= -0.2846
   SHORT: count= 26, mean=-0.0376, sum= -0.9768
   CLOSE: count= 35, mean=-0.0038, sum= -0.1330

RANDOM ACTION TEST SUMMARY
Total Steps: 100
Total Reward: -1.3937 | Avg/Step: -0.0139
Final Equity: $9,397.93
P&L: $-595.21

Action Distribution:
    HOLD:  24 ( 24.0%)
    LONG:  15 ( 15.0%)
   SHORT:  26 ( 26.0%)
   CLOSE:  35 ( 35.0%)

Reward by Action:
    HOLD: count= 24, mean= 0.0000, sum=  0.0007
    LONG: count= 15, mean=-0.0190, sum= -0.2846
   SHORT: count= 26, mean=-0.0376, sum= -0.9768
   CLOSE: count= 35, mean=-0.0038, sum= -0.1330



In [3]:
# Trade analysis - Show trade details across history
all_trades = history_records[-1].get('trades', [])
closed_trades = [t for t in all_trades if t.get('status') == 'CLOSED']

if closed_trades:
    print(f"\nTRADE ANALYSIS - All Closed Trades ({len(closed_trades)} total)")
    print(f"{'='*80}")
    
    # Show summary first
    if len(closed_trades) > 5:
        print(f"\nShowing first 3 and last 2 trades (out of {len(closed_trades)} total)...")
        trades_to_show = closed_trades[:3] + closed_trades[-2:]
    else:
        trades_to_show = closed_trades
    
    for trade_idx, trade in enumerate(trades_to_show, 1):
        # Find actual index in full list
        actual_idx = closed_trades.index(trade) + 1
        
        print(f"\nTrade #{actual_idx}")
        print(f"{'-'*80}")
        print(f"Direction: {'LONG' if trade['direction'] == 1 else 'SHORT'}")
        print(f"Entry Step: {trade['step_open']} | Exit Step: {trade['step_close']} | Duration: {trade.get('duration', 0)} steps")
        print(f"Entry: ${trade['entry_price']:,.2f} | Exit: ${trade.get('exit_price', 0):,.2f}")
        print(f"PnL: ${trade['pnl']:,.2f} ({trade['pnl_percent']*100:.2f}%) | Exit Reason: {trade['reason']}")
        
        # Show the trade progression across history steps
        step_open = trade['step_open']
        step_close = trade['step_close']
        
        # Only show progression for short trades or first few
        if trade.get('duration', 0) <= 10 or actual_idx <= 2:
            print(f"\nTrade progression (steps {step_open} to {step_close}):")
            print(f"{'Step':>4} | {'Price':>8} | {'Action':>6} | {'Equity':>10} | {'UnrPnL':>8} | {'UnrPnL%':>8}")
            print(f"{'-'*70}")
            
            action_names = {0: 'HOLD', 1: 'LONG', 2: 'SHORT', 3: 'CLOSE'}
            
            # Show relevant steps during this trade
            for i, record in enumerate(history_records):
                step = record['step']
                
                # Show steps within the trade range
                if step_open <= step <= step_close:
                    action_dir = record['action'][0] if isinstance(record['action'], (list, np.ndarray)) else record['action']
                    action_name = action_names.get(action_dir, str(action_dir))
                    price = record.get('current_price', 0)
                    equity = record['equity']
                    unrealized = record.get('unrealized_pnl', 0)
                    used_bal = record.get('used_balance', 0)
                    unrealized_pct = (unrealized / used_bal * 100) if used_bal > 0 else 0
                    
                    marker = ""
                    if step == step_open:
                        marker = " ← ENTRY"
                    elif step == step_close:
                        marker = " ← EXIT"
                    
                    print(f"{step:4d} | ${price:7.2f} | {action_name:>6} | ${equity:9.2f} | ${unrealized:7.2f} | {unrealized_pct:7.2f}%{marker}")
            
            print(f"{'-'*70}")
else:
    print("\nNo closed trades found.")


TRADE ANALYSIS - All Closed Trades (25 total)

Showing first 3 and last 2 trades (out of 25 total)...

Trade #1
--------------------------------------------------------------------------------
Direction: LONG
Entry Step: 288 | Exit Step: 292 | Duration: 4 steps
Entry: $113,575.36 | Exit: $113,487.65
PnL: $-33.34 (-9.72%) | Exit Reason: Manual Close

Trade progression (steps 288 to 292):
Step |    Price | Action |     Equity |   UnrPnL |  UnrPnL%
----------------------------------------------------------------------
 288 | $113575.36 |   LONG | $  9993.14 | $   0.00 |    0.00% ← ENTRY
 289 | $113474.06 |   HOLD | $  9962.55 | $ -30.59 |   -0.09%
 290 | $113551.60 |   LONG | $  9985.96 | $  -7.18 |   -0.02%
 291 | $113533.80 |   HOLD | $  9980.59 | $ -12.55 |   -0.04%
 292 | $113487.65 |  CLOSE | $  9959.80 | $   0.00 |    0.00% ← EXIT
----------------------------------------------------------------------

Trade #2
------------------------------------------------------------------------

## Test 2: Custom action sequence

In [4]:
# Reset environment for sequence test
env.reset()

# Define specific action sequence
action_sequence = (
    [[0, 0, 0]] * 2 +      # Wait 2 steps
    [[1, 4, 4]] +          # Enter LONG (RR=4.6x, ATR=2.4)
    [[0,0,0]] * 20 +     # Hold for 20 steps
    [[3, 0, 0]]            # Close position
)

print(f"Sequence: HOLD(2) → LONG → HOLD(20) → CLOSE")
print(f"{'='*80}\n")

history_records = []
for step, action in enumerate(action_sequence):
    obs, reward, done, truncated, info = env.step(action)
    history_records.append(info)
    if done or truncated:
        print(f"Episode ended at step {step}")
        break

df_history = pd.DataFrame(history_records)

# Summary
print(f"SEQUENCE TEST SUMMARY")
print(f"{'='*80}")
print(f"Steps: {len(df_history)} | Total Reward: {df_history['reward'].sum():.4f}")
print(f"Initial: ${df_history['equity'].iloc[0]:,.2f} | Final: ${df_history['equity'].iloc[-1]:,.2f}")
print(f"P&L: ${df_history['equity'].iloc[-1] - df_history['equity'].iloc[0]:,.2f} ({(df_history['equity'].iloc[-1] / df_history['equity'].iloc[0] - 1) * 100:.2f}%)")
print(f"{'='*80}\n")

# Show only steps with rewards != 0 or position changes
print("RELEVANT STEPS (Rewards & Position Changes):")
print(f"{'Step':>4} | {'Action':>6} | {'Reward':>8} | {'Equity':>10} | {'UnrPnL%':>8} | {'Note'}")
print(f"{'-'*80}")

action_names = {0: 'HOLD', 1: 'LONG', 2: 'SHORT', 3: 'CLOSE'}
prev_pos = 0

for i, row in df_history.iterrows():
    action_dir = row['action'][0] if isinstance(row['action'], (list, np.ndarray)) else row['action']
    action_name = action_names.get(action_dir, str(action_dir))
    pos_size = row['position_size']
    unrealized = row.get('unrealized_pnl', 0)
    used_bal = row.get('used_balance', 0)
    unrealized_pct = (unrealized / used_bal * 100) if used_bal > 0 else 0
    
    # Show if: reward != 0, position changed, or first/last step
    show_step = (
        row['reward'] != 0 or 
        pos_size != prev_pos or 
        i == 0 or 
        i == len(df_history) - 1
    )
    
    if show_step:
        note = ""
        if pos_size > 0 and prev_pos == 0:
            note = "ENTERED POSITION"
        elif pos_size == 0 and prev_pos > 0:
            note = "CLOSED POSITION"
        elif row['reward'] > 0:
            note = "Holding reward"
        
        print(f"{row['step']:4d} | {action_name:>6} | {row['reward']:8.4f} | ${row['equity']:9.2f} | {unrealized_pct:7.2f}% | {note}")
    
    prev_pos = pos_size

print(f"{'-'*80}\n")

# Trade analysis - Show trade details across history
all_trades = history_records[-1].get('trades', [])
closed_trades = [t for t in all_trades if t.get('status') == 'CLOSED']

if closed_trades:
    print(f"TRADE ANALYSIS - All Closed Trades")
    print(f"{'='*80}")
    
    for trade_idx, trade in enumerate(closed_trades, 1):
        print(f"\nTrade #{trade_idx}")
        print(f"{'-'*80}")
        print(f"Direction: {'LONG' if trade['direction'] == 1 else 'SHORT'}")
        print(f"Entry Step: {trade['step_open']} | Exit Step: {trade['step_close']} | Duration: {trade.get('duration', 0)} steps")
        print(f"Entry: ${trade['entry_price']:,.2f} | Exit: ${trade.get('exit_price', 0):,.2f}")
        print(f"PnL: ${trade['pnl']:,.2f} ({trade['pnl_percent']*100:.2f}%) | Exit Reason: {trade['reason']}")
        
        # Show the trade progression across history steps
        step_open = trade['step_open']
        step_close = trade['step_close']
        
        print(f"\nTrade progression (steps {step_open} to {step_close}):")
        print(f"{'Step':>4} | {'Price':>8} | {'Action':>6} | {'Equity':>10} | {'UnrPnL':>8} | {'UnrPnL%':>8} | {'Reward':>8}")
        print(f"{'-'*70}")
        
        # Show relevant steps during this trade
        for i, record in enumerate(history_records):
            step = record['step']
            
            # Show steps within the trade range
            if step_open <= step <= step_close:
                action_dir = record['action'][0] if isinstance(record['action'], (list, np.ndarray)) else record['action']
                action_name = action_names.get(action_dir, str(action_dir))
                price = record.get('current_price', 0)
                equity = record['equity']
                unrealized = record.get('unrealized_pnl', 0)
                used_bal = record.get('used_balance', 0)
                unrealized_pct = (unrealized / used_bal * 100) if used_bal > 0 else 0
                reward = (record.get('reward', -1))
                
                marker = ""
                if step == step_open:
                    marker = " ← ENTRY"
                elif step == step_close:
                    marker = " ← EXIT"
                
                print(f"{step:4d} | ${price:7.2f} | {action_name:>6} | ${equity:9.2f} | ${unrealized:7.2f} | {unrealized_pct:7.2f}% |{reward:.4f} {marker}")
        
        print(f"{'-'*70}")
else:
    print("No closed trades found.")

Sequence: HOLD(2) → LONG → HOLD(20) → CLOSE

SEQUENCE TEST SUMMARY
Steps: 24 | Total Reward: 0.0503
Initial: $10,000.00 | Final: $10,042.72
P&L: $42.72 (0.43%)

RELEVANT STEPS (Rewards & Position Changes):
Step | Action |   Reward |     Equity |  UnrPnL% | Note
--------------------------------------------------------------------------------
 288 |   HOLD |   0.0000 | $ 10000.00 |    0.00% | 
 290 |   LONG |  -0.0020 | $  9993.03 |    0.00% | ENTERED POSITION
 299 |   HOLD |   0.0001 | $ 10032.97 |    0.11% | Holding reward
 300 |   HOLD |   0.0001 | $ 10037.04 |    0.13% | Holding reward
 309 |   HOLD |   0.0001 | $ 10015.94 |    0.07% | Holding reward
 310 |   HOLD |   0.0001 | $ 10047.99 |    0.16% | Holding reward
 311 |  CLOSE |   0.0517 | $ 10042.72 |    0.00% | CLOSED POSITION
--------------------------------------------------------------------------------

TRADE ANALYSIS - All Closed Trades

Trade #1
-------------------------------------------------------------------------------